In [10]:
import numpy as np
datapath = "../../Preprocessing/Encoded_Data/ecoli_rna_embeddings.npz"
data = np.load(datapath)
embeddings = data["embeddings"]
labels = data["labels"]

In [11]:
counts = {}
for label in labels:
    counts[label] = counts[label] + 1 if label in counts else 1
print(counts)

{np.str_('rRNA'): 16, np.str_('tRNA'): 45, np.str_('sRNA'): 100}


In [12]:
mid_layer  = embeddings[:, 0, :]
last_layer = embeddings[:, 1, :]
embeddings_pooled = embeddings.mean(axis=1)
embeddings_pooled.shape

(161, 512)

In [13]:
print("Mid layer:", mid_layer[0][:10])
print("Last layer:", last_layer[0][:10])
print("Pooled: ", embeddings_pooled[0][:10])

Mid layer: [ 0.04351216 -1.0366505   0.84144366  1.6133971  -0.52261096  0.93100864
 -2.8923101  -0.44584432 -1.446854   -2.664081  ]
Last layer: [ 1.0053989  -0.8118206   0.636622    2.7043056  -0.29271957  0.8397843
 -2.9534838   1.4206294   2.317268    0.92349327]
Pooled:  [ 0.5244555  -0.9242356   0.73903286  2.1588514  -0.40766525  0.8853965
 -2.9228969   0.48739254  0.43520695 -0.8702939 ]


In [14]:
print(f"Number of clusters found: {len(set(labels)) - (1 if -1 in labels else 0)}")
print(f"Noise points: {sum(labels == -1)}")

Number of clusters found: 3
Noise points: 0


In [15]:
from sklearn.metrics import v_measure_score
from sklearn.cluster import HDBSCAN

embedding_options = {
    "Mid layer": mid_layer,
    "Last layer": last_layer,
    "Pooled": embeddings_pooled
}

best_per_embedding = {}
global_best_score = 0
global_best = {}

for embedding_name, embedding_data in embedding_options.items():
    best_score = 0
    best_params = {}
    best_results = {}

    for min_cluster_size in [5, 10, 20, 30, 50]:
        for min_samples in [1, 5, 10]:
            for metric in ["euclidean", "manhattan", "cosine"]:

                clusterer = HDBSCAN(
                    min_cluster_size=min_cluster_size,
                    min_samples=min_samples,
                    metric=metric,
                    copy=False
                )

                hdb_labels = clusterer.fit_predict(embedding_data)

                if len(set(hdb_labels)) <= 1:  # all noise or one cluster
                    continue

                score = v_measure_score(labels, hdb_labels)

                # best for THIS embedding
                if score > best_score:
                    best_score = score
                    best_params = {
                        'min_cluster_size': min_cluster_size,
                        'min_samples': min_samples,
                        'metric': metric
                    }
                    best_results = {
                        'num_clusters': len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0),
                        'num_noise': sum(hdb_labels == -1)
                    }

                # global best across all embeddings
                if score > global_best_score:
                    global_best_score = score
                    global_best = {
                        'embedding': embedding_name,
                        'min_cluster_size': min_cluster_size,
                        'min_samples': min_samples,
                        'metric': metric
                    }

    best_per_embedding[embedding_name] = {
        'score': best_score,
        'params': best_params,
        'results': best_results
    }

print("Best per embedding:")
for k, v in best_per_embedding.items():
    print(k, v)

print("\nGlobal best:")
print(global_best_score, global_best)

Best per embedding:
Mid layer {'score': 0.22243871176308355, 'params': {'min_cluster_size': 5, 'min_samples': 1, 'metric': 'euclidean'}, 'results': {'num_clusters': 4, 'num_noise': np.int64(82)}}
Last layer {'score': 0.21152120365479357, 'params': {'min_cluster_size': 5, 'min_samples': 1, 'metric': 'cosine'}, 'results': {'num_clusters': 3, 'num_noise': np.int64(89)}}
Pooled {'score': 0.16589503677454476, 'params': {'min_cluster_size': 5, 'min_samples': 5, 'metric': 'manhattan'}, 'results': {'num_clusters': 2, 'num_noise': np.int64(87)}}

Global best:
0.22243871176308355 {'embedding': 'Mid layer', 'min_cluster_size': 5, 'min_samples': 1, 'metric': 'euclidean'}


### Soft prediction for no -1 label

In [16]:
import hdbscan

embedding_options = {
    "Mid layer": mid_layer,
    "Last layer": last_layer,
    "Pooled": embeddings_pooled
}

best_per_embedding = {}
global_best_score = 0
global_best = {}

for embedding_name, embedding_data in embedding_options.items():
    best_score = 0
    best_params = {}
    best_results = {}

    for min_cluster_size in [5, 10, 20, 30, 50]:
        for min_samples in [1, 5, 10]:
            for metric in ["euclidean", "manhattan"]:

                clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=min_cluster_size,
                    min_samples=min_samples,
                    metric=metric,
                    prediction_data=True
                )

                hdb_labels = clusterer.fit_predict(embedding_data)

                if len(set(hdb_labels)) <= 1:  # all noise or one cluster
                    continue

                score = v_measure_score(labels, hdb_labels)

                # best for THIS embedding
                if score > best_score:
                    best_score = score
                    best_params = {
                        'min_cluster_size': min_cluster_size,
                        'min_samples': min_samples,
                        'metric': metric
                    }
                    best_results = {
                        'num_clusters': len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0),
                        'num_noise': sum(hdb_labels == -1)
                    }

                # global best across all embeddings
                if score > global_best_score:
                    global_best_score = score
                    global_best = {
                        'embedding': embedding_name,
                        'min_cluster_size': min_cluster_size,
                        'min_samples': min_samples,
                        'metric': metric
                    }

    best_per_embedding[embedding_name] = {
        'score': best_score,
        'params': best_params,
        'results': best_results
    }

print("Best per embedding:")
for k, v in best_per_embedding.items():
    print(k, v)

print("\nGlobal best:")
print(global_best_score, global_best)

Best per embedding:
Mid layer {'score': 0.22243871176308355, 'params': {'min_cluster_size': 5, 'min_samples': 1, 'metric': 'euclidean'}, 'results': {'num_clusters': 4, 'num_noise': np.int64(82)}}
Last layer {'score': 0.139371369644949, 'params': {'min_cluster_size': 5, 'min_samples': 1, 'metric': 'euclidean'}, 'results': {'num_clusters': 2, 'num_noise': np.int64(60)}}
Pooled {'score': 0.14861333065564641, 'params': {'min_cluster_size': 5, 'min_samples': 5, 'metric': 'manhattan'}, 'results': {'num_clusters': 2, 'num_noise': np.int64(93)}}

Global best:
0.22243871176308355 {'embedding': 'Mid layer', 'min_cluster_size': 5, 'min_samples': 1, 'metric': 'euclidean'}


In [18]:
import hdbscan

embedding_options = {
    "Mid layer": mid_layer,
    "Last layer": last_layer,
    "Pooled": embeddings_pooled
}

best_per_embedding = {}
global_best_score = 0
global_best = {}

for embedding_name, embedding_data in embedding_options.items():
    best_score = 0
    best_params = {}
    best_results = {}

    for min_cluster_size in [5, 10, 20, 30, 50]:
        for min_samples in [1, 5, 10]:
            for metric in ["euclidean", "manhattan"]:

                clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=min_cluster_size,
                    min_samples=min_samples,
                    metric=metric,
                    prediction_data=True
                )

                predictions = clusterer.fit_predict(embedding_data)

                soft_clusters = hdbscan.all_points_membership_vectors(clusterer)
                if soft_clusters.ndim == 1 or soft_clusters.shape[1] <= 1:
                    no_noise_predictions = predictions
                else:
                    best_cluster = np.argmax(soft_clusters, axis=1)
                    no_noise_predictions = np.where(predictions == -1, best_cluster, predictions)

                if len(set(no_noise_predictions)) <= 1:  # all noise or one cluster
                    continue

                score = v_measure_score(labels, no_noise_predictions)

                if score > best_score:
                    best_score = score
                    best_params = {
                        'min_cluster_size': min_cluster_size,
                        'min_samples': min_samples,
                        'metric': metric
                    }
                    best_results = {
                        'num_clusters': len(set(no_noise_predictions)) - (1 if -1 in no_noise_predictions else 0),
                        'num_noise': sum(no_noise_predictions == -1)
                    }

                if score > global_best_score:
                    global_best_score = score
                    global_best = {
                        'embedding': embedding_name,
                        'min_cluster_size': min_cluster_size,
                        'min_samples': min_samples,
                        'metric': metric
                    }

    best_per_embedding[embedding_name] = {
        'score': best_score,
        'params': best_params,
        'results': best_results
    }

print("Best per embedding:")
for k, v in best_per_embedding.items():
    print(k, v)

print("\nGlobal best:")
print(global_best_score, global_best)

Best per embedding:
Mid layer {'score': 0.1345092806856292, 'params': {'min_cluster_size': 5, 'min_samples': 1, 'metric': 'manhattan'}, 'results': {'num_clusters': 3, 'num_noise': np.int64(0)}}
Last layer {'score': 0.1163809590024271, 'params': {'min_cluster_size': 5, 'min_samples': 1, 'metric': 'manhattan'}, 'results': {'num_clusters': 2, 'num_noise': np.int64(0)}}
Pooled {'score': 0.1117739763573807, 'params': {'min_cluster_size': 5, 'min_samples': 5, 'metric': 'euclidean'}, 'results': {'num_clusters': 2, 'num_noise': np.int64(0)}}

Global best:
0.1345092806856292 {'embedding': 'Mid layer', 'min_cluster_size': 5, 'min_samples': 1, 'metric': 'manhattan'}
